In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
from IPython.display import Audio
from stable_baselines3 import PPO
# from sbx import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path
from stable_baselines3.common.monitor import Monitor
import sys
# from sbx.ppo.policies import CnnPolicy

import logging
logging.basicConfig(level=logging.DEBUG)


NUM_ENV = 1
SAVE_DIR="./model-gt3"
TENSORBOARD="./tensorboard-gt3"
TOTAL_TIMESTEP_NUMB = 500_000_000
CHECK_FREQ_NUMB = 5_000
SAVE_FREQ = CHECK_FREQ_NUMB
MAX_STEPS= 4_000

ENT_COEF = 0.001
n_steps=2048
batch_size=64 * NUM_ENV

SAVE_DIR = Path(SAVE_DIR)
combos = [
    [], # noop

    # # only turn without acelerate
    # ["LEFT"],
    # ["RIGHT"],

    # acelerate
    ['B'],
    ["LEFT", 'B'],
    ["RIGHT", 'B'],

    #  brake
    ["Y"],
    
    # reverse
    ["X"],
]


def make_env():
    def _init():
        env = make(
            "GranTurismo3-Ps2", 
            # render_mode="human"
        )
        # only dolphin wii need it
        # env.set_buttons(["B", "Y", "SELECT", "START", "LEFT", "RIGHT", "DOWN", "UP","A", "X", "L1", "R1", "L2", "R2", "L3", "R3"])

        env = MainDiscretizer(
            env,
            combos,
        )

        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=4)
        env = TimeLimit(env, max_steps=MAX_STEPS)

        return env
    return _init

    
# env = make_vec_env(make_env(), n_envs=NUM_ENV)

process_class = SubprocVecEnv
if NUM_ENV == 1:
    process_class = DummyVecEnv

# envs = [make_env(i) for i in range(NUM_ENV)]
# env = SubprocVecEnv(envs)
env = make_vec_env(make_env(), n_envs=NUM_ENV, vec_env_cls=process_class)
env = VecFrameStack(env, 4, channels_order='last')

latest_model_path = get_latest_model(SAVE_DIR)

if latest_model_path:
    print(f"Loading existent model: {latest_model_path}")
    model = PPO.load(
    # model = RecurrentPPO.load(
        str(latest_model_path), 
        env=env, 
        verbose=0, 
        tensorboard_log=TENSORBOARD, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
    )
    
else:
    print("None finded, starting from zero.")
    model = PPO("CnnPolicy", 
    # model = RecurrentPPO('CnnLstmPolicy',
        env, 
        verbose=0, 
        # policy_kwargs=policy_kwargs, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
        tensorboard_log=TENSORBOARD, 
    )

# eval_callback = EvalCallback(
#     eval_env, 
#     best_model_save_path="./logs/best_model",
#     log_path="./logs/results", 
#     eval_freq=5_000,
#     n_eval_episodes=6,
#     deterministic=True
# )

checkpoint_callback=TrainAndLoggingCallback(check_freq=CHECK_FREQ_NUMB, save_path=SAVE_DIR, save_freq=SAVE_FREQ, model=model)
#callback = CallbackList([checkpoint_callback, eval_callback])
callback = CallbackList([checkpoint_callback])

model.learn(total_timesteps=TOTAL_TIMESTEP_NUMB, reset_num_timesteps=False, callback=callback)
model.save("final_gt3")

env.close()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
None finded, starting from zero.
Done Rewards Step Cnt: 2688
Model saved in: model-gt3\best_model_5000
Done Rewards Step Cnt: 2579
Done Rewards Step Cnt: 2529
Model saved in: model-gt3\best_model_10000
Done Rewards Step Cnt: 2529
Done Rewards Step Cnt: 2527
Model saved in: model-gt3\best_model_15000
Done Rewards Step Cnt: 2558
Done Rewards Step Cnt: 2510
Model saved in: model-gt3\best_model_20000
Done Rewards Step Cnt: 2539
Done Rewards Step Cnt: 2512
Model saved in: model-gt3\best_model_25000
Done Rewards Step Cnt: 2610
Done Rewards Step Cnt: 2689
Model saved in: model-gt3\best_model_30000
Done Rewards Step Cnt: 2513
Done Rewards Step Cnt: 2589
Model saved in: model-gt3\best_model_35000
Done Rewards Step Cnt: 2518
Done Rewards Step Cnt: 2508
Model saved in: model-gt3\best_model_40000
Done Rewards Step Cnt: 2511
Done Rewards Step Cnt: 2499
Model saved in: model-gt3\best_model_45000
Done Rewards Step Cnt: 2512
Done Rewards Step Cnt: 2522
Model 